# 4- vs 12-carrier timing of frozen generic+R, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

Keep the candidate frozen. Do **not** load `plan4_dev` or `plan4_fresh` for retuning. Do **not** overwrite `results_stage_plan4_hold` or `results_stage_plan4_confirm`. Do not regenerate hold A. Do not train an adapter. Do not add calculations. Do not search more support variants.

**Accuracy stays the hold confirmation** (previously evaluated `plan4_hold` families, not untouched):
- result-only KV 19/90 labels, 43/90 choices
- generic+R 83/90 labels, 77/90 choices
- full KV 88/90 labels, 79/90 choices
- result text 87/90 labels, 74/90 choices

The candidate is 5.56 points below full KV and 4.44 below text. Paired intervals include gaps larger than five points, so recovery to both references is unestablished.

This run measures costs at **4 and 12 carriers**, separately:
1. message preparation (rebuild / construct; generic support prepared once per template and reused)
2. receiver prefill of both B calls
3. decoding of both B calls

Cache figures are packed-state mebibytes, not measured network traffic.

**Cell 4** (`--phase check`) reconstructs 9 hold base families.
**Cell 5** (`--phase write12`) writes 12-carrier A on the timing-only split.
**Cell 6** (`--phase compare`) times B.

Upload `src/`, `data/plan4_hold.jsonl`, `data/plan4_timing12.jsonl`, and `results_stage_plan4_hold/a_artifacts.jsonl`.

Download `/kaggle/working/results_stage_plan4_timing`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_timing.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_timing.py").is_file() and (cand / "data" / "plan4_hold.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_timing.py and data/plan4_hold.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan4_timing.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
art = REPO / "results_stage_plan4_hold" / "a_artifacts.jsonl"
print("frozen hold A artifacts", art.is_file(), art)
print("timing12", (REPO / "data" / "plan4_timing12.jsonl").is_file())
print("do not overwrite", WORK / "results_stage_plan4_hold", WORK / "results_stage_plan4_confirm")

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_timing")
print("Check 4-carrier construction, write 12-carrier A, then compare.")
print("Do not overwrite results_stage_plan4_hold or results_stage_plan4_confirm.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_timing", "--split", "plan4_timing", "--phase", "check"])
print("plan4 timing check exit", rc)
path = WORK / "results_stage_plan4_timing" / "check_summary.json"
if not path.is_file():
    path = REPO / "results" / "stage_plan4_timing" / "run" / "check_summary.json"
if path.is_file():
    data = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps({"n": data.get("n"), "all_passed": data.get("all_passed"), "generic_prepare_once": data.get("generic_prepare_once")}, indent=2)[:8000])

In [ ]:
src = WORK / "results_stage_plan4_timing" / "check_summary.json"
if not src.is_file():
    src = REPO / "results" / "stage_plan4_timing" / "run" / "check_summary.json"
check = json.loads(src.read_text(encoding="utf-8"))
if not check.get("all_passed"):
    raise RuntimeError("4-carrier construction failed. Do not write 12-carrier A.")
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_timing", "--split", "plan4_timing", "--phase", "write12"])
print("plan4 timing write12 exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_timing" / "check12_summary.json"
if not src.is_file():
    src = REPO / "results" / "stage_plan4_timing" / "run" / "check12_summary.json"
check12 = json.loads(src.read_text(encoding="utf-8"))
if not check12.get("all_passed"):
    raise RuntimeError("12-carrier construction failed. Do not run compare.")
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_timing", "--split", "plan4_timing", "--phase", "compare"])
print("plan4 timing compare exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_timing"
if not (src / "summary.json").is_file() and not (src / "check_summary.json").is_file():
    src = REPO / "results" / "stage_plan4_timing" / "run"
print("reading", src)
for name in ("check_summary.json", "check12_summary.json", "summary.json", "timing_summary.json"):
    path = src / name
    if not path.is_file():
        continue
    data = json.loads(path.read_text(encoding="utf-8"))
    print("\n====", name, "====")
    keep = {k: data.get(k) for k in ("n", "n_rows", "all_passed", "confirm_accuracy", "generic_prepare_once", "by_n_carriers", "lead") if k in data or data.get(k) is not None}
    if "by_n_carriers" in keep and isinstance(keep["by_n_carriers"], dict):
        slim = {}
        for width, block in keep["by_n_carriers"].items():
            recv = block.get("receiver") or {}
            slim[width] = {
                "n_examples": block.get("n_examples"),
                "cache_mib": block.get("cache_mib"),
                "receiver": {
                    cond: {kk: (recv.get(cond) or {}).get(kk) for kk in (
                        "n", "parse_ok", "truncated", "plan_all_status_correct", "correct",
                        "mean_relay_mb", "mean_prep_s", "mean_b_prefill_s", "mean_b_decode_s",
                        "mean_process_time_s", "mean_process_time_amortized_s",
                        "mean_generic_prefill_s",
                    )}
                    for cond in recv
                },
            }
        keep["by_n_carriers"] = slim
    print(json.dumps(keep, indent=2)[:24000])